# Computational Notebook 12: Governance Simulation

## Overview

On-chain governance is the mechanism by which decentralized protocols make collective decisions -- upgrading code, allocating treasury funds, adjusting parameters, and resolving disputes -- without relying on a central authority. This notebook builds working simulations of governance primitives: proposal lifecycles, voting mechanisms (token-weighted, quadratic, conviction), delegation systems, governance attack vectors, treasury management, and quantitative governance health metrics. Every model is implemented in pure Python so you can experiment with parameters and observe how design choices affect governance outcomes.

## Prerequisites
- **Notebook 04**: Smart Contract Development (contract mechanics, token standards)
- **Notebook 10**: Cryptoeconomic Modeling (game theory, mechanism design)
- **Notebook 11**: Tokenomics (token distribution, vesting, ve-tokens)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Simulate the full governance proposal lifecycle: propose, vote, queue, execute
2. Implement and compare token-weighted, quadratic, and conviction voting mechanisms
3. Model governance attack vectors including flash loan attacks and vote buying
4. Build a vote delegation system and analyze power concentration
5. Simulate DAO treasury management with budget proposals and spending limits
6. Calculate governance health metrics: participation rates, Gini coefficient, Nakamoto coefficient

**Estimated Time:** 4-6 hours

**Related Content:** [Section 08: DAO Governance](../sections/08-dao-governance.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
from enum import Enum
import hashlib
import time

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook simulates on-chain governance mechanisms.")

---
## 1. Governance Fundamentals

On-chain governance follows a structured lifecycle:

1. **Proposal Creation** -- A token holder submits a proposal (requires minimum tokens)
2. **Voting Period** -- Token holders vote For, Against, or Abstain
3. **Quorum Check** -- Minimum participation threshold must be met
4. **Timelock** -- Successful proposals enter a delay period before execution
5. **Execution** -- The proposal's on-chain actions are executed

> **Definition: Quorum** -- The minimum number of votes (or percentage of total supply) required for a governance vote to be valid. Prevents small minorities from passing proposals when participation is low.

> **Definition: Timelock** -- A mandatory delay between a proposal's approval and its execution, giving the community time to react (e.g., exit the protocol) if they disagree.

**Source:** OpenZeppelin. (2023). *Governor Contract Documentation*. docs.openzeppelin.com

In [ ]:
class ProposalState(Enum):
    """States in the governance proposal lifecycle."""
    PENDING = "Pending"       # Created, voting not started
    ACTIVE = "Active"         # Voting in progress
    DEFEATED = "Defeated"     # Failed vote or quorum not met
    SUCCEEDED = "Succeeded"   # Passed vote and quorum met
    QUEUED = "Queued"         # In timelock waiting period
    EXECUTED = "Executed"     # Successfully executed
    EXPIRED = "Expired"       # Timelock expired without execution


@dataclass
class Proposal:
    """A governance proposal."""
    id: int
    proposer: str
    description: str
    created_at: int           # Block number
    voting_start: int
    voting_end: int
    votes_for: float = 0.0
    votes_against: float = 0.0
    votes_abstain: float = 0.0
    state: ProposalState = ProposalState.PENDING
    voters: Dict[str, str] = field(default_factory=dict)  # address -> vote
    queued_at: Optional[int] = None
    executed_at: Optional[int] = None


class GovernorSimulator:
    """OpenZeppelin Governor-style governance simulator."""
    
    def __init__(self, token_balances: Dict[str, float],
                 quorum_pct: float = 4.0,
                 voting_period: int = 50400,  # ~7 days in blocks
                 voting_delay: int = 7200,    # ~1 day delay
                 timelock_delay: int = 14400,  # ~2 days
                 proposal_threshold: float = 0.01) -> None:
        """Initialize governor.
        
        Args:
            token_balances: Address -> token balance mapping
            quorum_pct: Percentage of total supply needed for quorum
            voting_period: Number of blocks voting is open
            voting_delay: Blocks between proposal and voting start
            timelock_delay: Blocks between success and execution
            proposal_threshold: Min fraction of supply to propose
        """
        self.balances = dict(token_balances)
        self.total_supply = sum(token_balances.values())
        self.quorum = self.total_supply * quorum_pct / 100
        self.voting_period = voting_period
        self.voting_delay = voting_delay
        self.timelock_delay = timelock_delay
        self.proposal_threshold = proposal_threshold * self.total_supply
        self.proposals: Dict[int, Proposal] = {}
        self.next_id = 1
        self.current_block = 0
        self.event_log: List[str] = []
    
    def advance_blocks(self, n: int) -> None:
        """Advance the block number."""
        self.current_block += n
        # Update proposal states
        for p in self.proposals.values():
            if p.state == ProposalState.PENDING and self.current_block >= p.voting_start:
                p.state = ProposalState.ACTIVE
            elif p.state == ProposalState.ACTIVE and self.current_block >= p.voting_end:
                total_votes = p.votes_for + p.votes_against + p.votes_abstain
                if total_votes >= self.quorum and p.votes_for > p.votes_against:
                    p.state = ProposalState.SUCCEEDED
                else:
                    p.state = ProposalState.DEFEATED
    
    def propose(self, proposer: str, description: str) -> int:
        """Create a new proposal."""
        if self.balances.get(proposer, 0) < self.proposal_threshold:
            raise ValueError(f"{proposer} has insufficient tokens to propose "
                           f"(has {self.balances.get(proposer, 0):,.0f}, "
                           f"needs {self.proposal_threshold:,.0f})")
        
        pid = self.next_id
        self.next_id += 1
        
        proposal = Proposal(
            id=pid, proposer=proposer, description=description,
            created_at=self.current_block,
            voting_start=self.current_block + self.voting_delay,
            voting_end=self.current_block + self.voting_delay + self.voting_period
        )
        self.proposals[pid] = proposal
        self.event_log.append(f"Block {self.current_block}: Proposal #{pid} created by {proposer}")
        return pid
    
    def cast_vote(self, voter: str, proposal_id: int, support: str) -> None:
        """Cast a vote on a proposal.
        
        Args:
            voter: Address of the voter
            proposal_id: ID of the proposal
            support: 'for', 'against', or 'abstain'
        """
        p = self.proposals[proposal_id]
        
        if p.state != ProposalState.ACTIVE:
            raise ValueError(f"Proposal #{proposal_id} is not active (state: {p.state.value})")
        if voter in p.voters:
            raise ValueError(f"{voter} already voted on proposal #{proposal_id}")
        
        weight = self.balances.get(voter, 0)
        if weight == 0:
            raise ValueError(f"{voter} has no voting power")
        
        p.voters[voter] = support
        if support == 'for':
            p.votes_for += weight
        elif support == 'against':
            p.votes_against += weight
        else:
            p.votes_abstain += weight
        
        self.event_log.append(
            f"Block {self.current_block}: {voter} voted {support} on #{proposal_id} "
            f"(weight: {weight:,.0f})")
    
    def queue(self, proposal_id: int) -> None:
        """Queue a succeeded proposal for execution."""
        p = self.proposals[proposal_id]
        if p.state != ProposalState.SUCCEEDED:
            raise ValueError(f"Proposal #{proposal_id} has not succeeded")
        p.state = ProposalState.QUEUED
        p.queued_at = self.current_block
        self.event_log.append(f"Block {self.current_block}: Proposal #{proposal_id} queued")
    
    def execute(self, proposal_id: int) -> None:
        """Execute a queued proposal after timelock."""
        p = self.proposals[proposal_id]
        if p.state != ProposalState.QUEUED:
            raise ValueError(f"Proposal #{proposal_id} is not queued")
        if self.current_block < p.queued_at + self.timelock_delay:
            raise ValueError(f"Timelock not expired (need block {p.queued_at + self.timelock_delay})")
        p.state = ProposalState.EXECUTED
        p.executed_at = self.current_block
        self.event_log.append(f"Block {self.current_block}: Proposal #{proposal_id} EXECUTED")


# Set up a DAO with realistic token distribution
np.random.seed(42)
n_holders = 1000
total_supply = 100_000_000  # 100M tokens

# Power-law distribution (few whales, many small holders)
raw = np.random.pareto(1.5, n_holders)
balances_arr = raw / raw.sum() * total_supply
balances_arr.sort()  # Sort ascending
balances_arr = balances_arr[::-1]  # Descending

token_balances = {f"holder_{i:04d}": bal for i, bal in enumerate(balances_arr)}

gov = GovernorSimulator(token_balances, quorum_pct=4.0)

print("=" * 60)
print("GOVERNOR SIMULATOR")
print("=" * 60)
print(f"Total supply: {total_supply:,} tokens")
print(f"Token holders: {n_holders}")
print(f"Quorum: {gov.quorum:,.0f} tokens ({gov.quorum/total_supply*100:.1f}%)")
print(f"Proposal threshold: {gov.proposal_threshold:,.0f} tokens")
print(f"\nTop 10 holders:")
for i in range(10):
    name = f"holder_{i:04d}"
    bal = token_balances[name]
    print(f"  {name}: {bal:>12,.0f} ({bal/total_supply*100:.2f}%)")

In [ ]:
# Run a full proposal lifecycle
print("=" * 60)
print("PROPOSAL LIFECYCLE SIMULATION")
print("=" * 60)

# Step 1: Create proposal
pid = gov.propose("holder_0000", "Increase staking rewards from 5% to 8%")
print(f"\n1. Proposal #{pid} created at block {gov.current_block}")
print(f"   State: {gov.proposals[pid].state.value}")

# Step 2: Advance to voting period
gov.advance_blocks(gov.voting_delay + 1)
print(f"\n2. Advanced to block {gov.current_block} (voting started)")
print(f"   State: {gov.proposals[pid].state.value}")

# Step 3: Cast votes
for_voters = [f"holder_{i:04d}" for i in range(0, 50)]   # Top 50 vote for
against_voters = [f"holder_{i:04d}" for i in range(50, 80)]  # Next 30 vote against
abstain_voters = [f"holder_{i:04d}" for i in range(80, 100)]  # Next 20 abstain

for v in for_voters:
    gov.cast_vote(v, pid, 'for')
for v in against_voters:
    gov.cast_vote(v, pid, 'against')
for v in abstain_voters:
    gov.cast_vote(v, pid, 'abstain')

p = gov.proposals[pid]
total_voted = p.votes_for + p.votes_against + p.votes_abstain
print(f"\n3. Votes cast: {len(p.voters)} voters")
print(f"   For:     {p.votes_for:>12,.0f} ({p.votes_for/total_voted*100:.1f}%)")
print(f"   Against: {p.votes_against:>12,.0f} ({p.votes_against/total_voted*100:.1f}%)")
print(f"   Abstain: {p.votes_abstain:>12,.0f} ({p.votes_abstain/total_voted*100:.1f}%)")
print(f"   Quorum:  {total_voted:>12,.0f} / {gov.quorum:,.0f} ({'MET' if total_voted >= gov.quorum else 'NOT MET'})")

# Step 4: End voting
gov.advance_blocks(gov.voting_period)
print(f"\n4. Voting ended at block {gov.current_block}")
print(f"   State: {gov.proposals[pid].state.value}")

# Step 5: Queue
gov.queue(pid)
print(f"\n5. Proposal queued at block {gov.current_block}")
print(f"   State: {gov.proposals[pid].state.value}")

# Step 6: Execute after timelock
gov.advance_blocks(gov.timelock_delay + 1)
gov.execute(pid)
print(f"\n6. Proposal executed at block {gov.current_block}")
print(f"   State: {gov.proposals[pid].state.value}")
print(f"\n   Total lifecycle: {gov.current_block} blocks")

---
## 2. Voting Mechanisms

Different voting mechanisms produce different outcomes and have different resistance to attack.

### Token-Weighted Voting
1 token = 1 vote. Simple but plutocratic -- whales dominate.

### Quadratic Voting (QV)
Voting power = $\sqrt{\text{tokens}}$. Reduces whale dominance while still rewarding larger stakes.

> **Definition: Quadratic Voting (QV)** -- A voting mechanism where the cost of additional votes increases quadratically: 1 vote costs 1 token, 2 votes cost 4, 3 votes cost 9, etc. Equivalently, voting power equals the square root of tokens committed.

### Conviction Voting
Voting power accumulates over time. Longer commitment = more influence. Prevents last-minute vote swings.

$$\text{conviction}(t) = \text{tokens} \cdot (1 - \alpha^t)$$

where $\alpha$ is the decay factor (e.g., 0.9) and $t$ is time periods staked.

**Source:** Buterin, V. (2019). "Quadratic Payments: A Primer." vitalik.eth.limo

In [ ]:
def token_weighted_vote(balances: Dict[str, float], votes: Dict[str, str]) -> Dict[str, float]:
    """Standard 1-token-1-vote."""
    results = {'for': 0, 'against': 0, 'abstain': 0}
    for voter, choice in votes.items():
        results[choice] += balances.get(voter, 0)
    return results


def quadratic_vote(balances: Dict[str, float], votes: Dict[str, str]) -> Dict[str, float]:
    """Quadratic voting: power = sqrt(tokens)."""
    results = {'for': 0, 'against': 0, 'abstain': 0}
    for voter, choice in votes.items():
        results[choice] += np.sqrt(balances.get(voter, 0))
    return results


def conviction_vote(balances: Dict[str, float], votes: Dict[str, str],
                    stake_duration: Dict[str, int], alpha: float = 0.9) -> Dict[str, float]:
    """Conviction voting: power grows with time staked."""
    results = {'for': 0, 'against': 0, 'abstain': 0}
    for voter, choice in votes.items():
        tokens = balances.get(voter, 0)
        t = stake_duration.get(voter, 1)
        conviction = tokens * (1 - alpha ** t)
        results[choice] += conviction
    return results


# Compare voting mechanisms on same vote
np.random.seed(123)

# Scenario: 5 whales vote against, 200 small holders vote for
test_balances = {}
test_votes = {}
test_duration = {}

# 5 whales (1M tokens each)
for i in range(5):
    name = f"whale_{i}"
    test_balances[name] = 1_000_000
    test_votes[name] = 'against'
    test_duration[name] = 3  # Recent arrivals

# 200 community members (5,000 tokens each)
for i in range(200):
    name = f"community_{i}"
    test_balances[name] = 5_000
    test_votes[name] = 'for'
    test_duration[name] = 30  # Long-term members

whale_total = 5 * 1_000_000
community_total = 200 * 5_000

print("=" * 60)
print("VOTING MECHANISM COMPARISON")
print("=" * 60)
print(f"\nScenario: 5 whales (1M each) vs 200 community (5K each)")
print(f"Whale tokens: {whale_total:,} | Community tokens: {community_total:,}")
print(f"Whales vote: AGAINST | Community votes: FOR")

mechanisms = [
    ("Token-Weighted", token_weighted_vote(test_balances, test_votes)),
    ("Quadratic", quadratic_vote(test_balances, test_votes)),
    ("Conviction", conviction_vote(test_balances, test_votes, test_duration)),
]

print(f"\n{'Mechanism':<18} {'FOR':>12} {'AGAINST':>12} {'Winner':>10}")
print("-" * 55)
for name, result in mechanisms:
    winner = 'FOR' if result['for'] > result['against'] else 'AGAINST'
    print(f"{name:<18} {result['for']:>12,.0f} {result['against']:>12,.0f} {winner:>10}")

In [ ]:
# Visualize voting power distribution under each mechanism
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Generate voting power for all holders
all_balances = sorted(test_balances.values(), reverse=True)

# Token-weighted
tw_power = np.array(all_balances)
tw_cumulative = np.cumsum(tw_power) / tw_power.sum() * 100

# Quadratic
qv_power = np.sqrt(np.array(all_balances))
qv_cumulative = np.cumsum(qv_power) / qv_power.sum() * 100

# Conviction (assume uniform 10 periods)
cv_power = np.array(all_balances) * (1 - 0.9 ** 10)
cv_cumulative = np.cumsum(cv_power) / cv_power.sum() * 100

holder_pct = np.arange(1, len(all_balances) + 1) / len(all_balances) * 100

data = [
    ("Token-Weighted (1:1)", tw_cumulative),
    ("Quadratic (sqrt)", qv_cumulative),
    ("Conviction (time-weighted)", cv_cumulative),
]

for ax, (title, cum) in zip(axes, data):
    ax.plot(holder_pct, cum, 'b-', linewidth=2)
    ax.plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect equality')
    ax.fill_between(holder_pct, cum, holder_pct, alpha=0.2)
    ax.set_xlabel('% of Holders (ranked by power)')
    ax.set_ylabel('% of Voting Power')
    ax.set_title(title)
    ax.legend(fontsize=8)
    
    # Annotate top 5 holders' power
    top5_pct = cum[4]
    ax.annotate(f'Top 5 holders: {top5_pct:.1f}%',
                xy=(2.4, top5_pct), fontsize=8,
                xytext=(30, -20), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='red'))

plt.suptitle('Lorenz Curves: Voting Power Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/voting_mechanisms.png', dpi=100, bbox_inches='tight')
plt.show()
print("Quadratic voting significantly flattens the power distribution.")

---
## 3. Governance Attack Vectors

On-chain governance systems face several attack vectors:

### Flash Loan Governance Attack
An attacker borrows a large number of governance tokens via a flash loan, votes on a malicious proposal, and returns the tokens -- all in one transaction.

### Vote Buying
An attacker pays token holders to delegate their votes or vote a certain way, effectively purchasing governance power without buying tokens.

### Voter Apathy Exploitation
When participation is low, a relatively small stake can pass proposals if the quorum is low.

**Source:** Daian, P. et al. (2020). "Flash Boys 2.0." *IEEE S&P*.

In [ ]:
def simulate_flash_loan_attack(total_supply: float, quorum_pct: float,
                                typical_participation: float,
                                uses_snapshot: bool = False) -> Dict:
    """Simulate a flash loan governance attack.
    
    Args:
        total_supply: Total governance token supply
        quorum_pct: Required quorum as percentage
        typical_participation: Typical voter turnout as fraction
        uses_snapshot: Whether the protocol snapshots balances before voting
    """
    quorum = total_supply * quorum_pct / 100
    typical_votes = total_supply * typical_participation
    
    # Without snapshot: attacker can borrow and vote in same block
    if not uses_snapshot:
        # Need more votes than typical participation + quorum
        attack_tokens = max(quorum, typical_votes) * 1.1  # 10% margin
        attack_cost = attack_tokens * 0.0009  # Flash loan fee ~0.09%
        success = True
    else:
        # With snapshot: must hold tokens BEFORE proposal created
        attack_tokens = max(quorum, typical_votes) * 1.1
        attack_cost = attack_tokens  # Must actually buy tokens
        success = False  # Much harder
    
    return {
        "tokens_needed": attack_tokens,
        "attack_cost": attack_cost,
        "success_likely": success,
        "uses_snapshot": uses_snapshot,
        "quorum": quorum,
        "typical_votes": typical_votes
    }


print("=" * 60)
print("GOVERNANCE ATTACK ANALYSIS")
print("=" * 60)

# Scenario: DAO with 100M tokens, $1 each
supply = 100_000_000
participation = 0.08  # 8% typical turnout

print(f"\nDAO: 100M tokens at $1 each")
print(f"Typical voter turnout: {participation:.0%}")

print(f"\n{'Quorum':>8} {'Snapshot':>10} {'Tokens Needed':>15} {'Attack Cost':>14} {'Likely?':>10}")
print("-" * 60)

for quorum_pct in [2, 4, 10, 20]:
    for snapshot in [False, True]:
        result = simulate_flash_loan_attack(supply, quorum_pct, participation, snapshot)
        print(f"{quorum_pct:>7}% {'Yes':>10 if snapshot else 'No':>10} "
              f"{result['tokens_needed']:>13,.0f} ${result['attack_cost']:>12,.0f} "
              f"{'Yes' if result['success_likely'] else 'No':>10}")

print(f"\nKey defense: Snapshot token balances BEFORE proposal creation.")
print(f"This makes flash loan attacks infeasible (must buy tokens, not borrow).")

In [ ]:
# Simulate vote buying economics
print("\n" + "=" * 60)
print("VOTE BUYING ANALYSIS")
print("=" * 60)

# How much does it cost to buy enough votes?
token_price = 1.0
supply = 100_000_000
participation_rates = [0.05, 0.10, 0.20, 0.40]
bribe_per_token = [0.01, 0.05, 0.10]  # $/token bribe

print(f"\nCost to buy a majority vote at different participation & bribe rates:")
print(f"\n{'Participation':>14} {'Votes to Win':>14}", end='')
for bribe in bribe_per_token:
    print(f" {'$'+f'{bribe:.2f}/tok':>14}", end='')
print()
print("-" * 58)

for part in participation_rates:
    votes_needed = supply * part * 0.51  # Need 51% of participating votes
    print(f"{part:>13.0%} {votes_needed:>13,.0f}", end='')
    for bribe in bribe_per_token:
        cost = votes_needed * bribe
        print(f" ${cost:>13,.0f}", end='')
    print()

print(f"\nDefenses: time-locked voting, vote escrow (veTokens), reputation systems")

---
## 4. Delegate Systems

Most governance token holders never vote directly. Delegation allows passive holders to assign their voting power to active participants (delegates).

> **Definition: Vote Delegation** -- The act of assigning one's governance voting power to another address (a delegate) who votes on their behalf. The delegator retains ownership of tokens but transfers voting rights.

Delegation creates power concentration that can be measured and monitored.

In [ ]:
class DelegationSystem:
    """Token delegation system for governance."""
    
    def __init__(self, balances: Dict[str, float]) -> None:
        """Initialize with token balances."""
        self.balances = dict(balances)
        self.delegations: Dict[str, str] = {}  # delegator -> delegate
    
    def delegate(self, delegator: str, delegate: str) -> None:
        """Delegate voting power to another address."""
        if delegator not in self.balances:
            raise ValueError(f"{delegator} has no tokens")
        self.delegations[delegator] = delegate
    
    def get_voting_power(self, address: str) -> float:
        """Get total voting power including delegations."""
        own = self.balances.get(address, 0)
        # Don't count own balance if delegated away
        if address in self.delegations and self.delegations[address] != address:
            own = 0
        delegated = sum(
            self.balances[d] for d, target in self.delegations.items()
            if target == address and d != address
        )
        return own + delegated
    
    def get_all_voting_power(self) -> Dict[str, float]:
        """Get voting power for all addresses with non-zero power."""
        powers = {}
        all_addresses = set(self.balances.keys()) | set(self.delegations.values())
        for addr in all_addresses:
            power = self.get_voting_power(addr)
            if power > 0:
                powers[addr] = power
        return powers


# Simulate delegation patterns
np.random.seed(42)

# Use our earlier token distribution
deleg_system = DelegationSystem(token_balances)

# 10 active delegates
delegates = [f"holder_{i:04d}" for i in range(10)]

# 70% of holders delegate to one of the top 10
for i in range(10, n_holders):
    holder = f"holder_{i:04d}"
    if np.random.random() < 0.70:  # 70% delegate
        # Prefer top delegates (weighted random)
        weights = np.array([token_balances[d] for d in delegates])
        weights = weights / weights.sum()
        chosen = np.random.choice(delegates, p=weights)
        deleg_system.delegate(holder, chosen)

# Get voting power distribution
powers = deleg_system.get_all_voting_power()
sorted_powers = sorted(powers.items(), key=lambda x: x[1], reverse=True)

print("=" * 60)
print("DELEGATION SYSTEM ANALYSIS")
print("=" * 60)

print(f"\nActive delegates: {len([p for _, p in sorted_powers if p > 0])}")
print(f"Delegated tokens: {sum(token_balances[d] for d in deleg_system.delegations):,.0f}")
print(f"\nTop 10 delegates by voting power:")
total_power = sum(p for _, p in sorted_powers)
cumulative = 0
for name, power in sorted_powers[:10]:
    cumulative += power
    print(f"  {name}: {power:>12,.0f} ({power/total_power*100:>5.1f}%) "
          f"[cumulative: {cumulative/total_power*100:.1f}%]")

---
## 5. DAO Treasury Management

DAOs control treasuries worth billions of dollars. Treasury management involves:
- Budget allocation across teams and initiatives
- Spending limits and approval thresholds
- Diversification (not holding 100% native token)
- Runway management (months of operating expenses)

In [ ]:
@dataclass
class TreasuryProposal:
    """A treasury spending proposal."""
    id: int
    title: str
    amount_usd: float
    category: str
    recipient: str
    approved: bool = False


class DAOTreasury:
    """DAO treasury management simulator."""
    
    def __init__(self, holdings: Dict[str, Tuple[float, float]]) -> None:
        """Initialize treasury.
        
        Args:
            holdings: {asset_name: (amount, price_usd)}
        """
        self.holdings = dict(holdings)
        self.spending_history: List[TreasuryProposal] = []
        self.monthly_burn = 0.0
    
    def total_value(self) -> float:
        """Calculate total treasury value in USD."""
        return sum(amt * price for amt, price in self.holdings.values())
    
    def diversification_score(self) -> float:
        """Calculate diversification (1 - HHI). Higher = more diversified."""
        total = self.total_value()
        if total == 0:
            return 0
        shares = [(amt * price / total) for amt, price in self.holdings.values()]
        hhi = sum(s ** 2 for s in shares)
        return 1 - hhi
    
    def runway_months(self, monthly_expenses: float) -> float:
        """Calculate treasury runway in months."""
        return self.total_value() / monthly_expenses if monthly_expenses > 0 else float('inf')
    
    def simulate_market_impact(self, native_token: str,
                                price_changes: Dict[str, float]) -> float:
        """Simulate treasury value after price changes."""
        new_total = 0
        for asset, (amt, price) in self.holdings.items():
            change = price_changes.get(asset, 0)
            new_total += amt * price * (1 + change)
        return new_total


# Example: Large DAO treasury
treasury = DAOTreasury({
    "GOV_TOKEN": (50_000_000, 2.50),   # 50M tokens at $2.50
    "ETH": (5_000, 2_000),             # 5,000 ETH
    "USDC": (15_000_000, 1.00),        # $15M stables
    "USDT": (5_000_000, 1.00),         # $5M stables
})

monthly_expenses = 2_000_000  # $2M/month

print("=" * 60)
print("DAO TREASURY ANALYSIS")
print("=" * 60)

print(f"\n{'Asset':<15} {'Amount':>14} {'Price':>10} {'Value':>14} {'Share':>8}")
print("-" * 65)
total = treasury.total_value()
for asset, (amt, price) in treasury.holdings.items():
    value = amt * price
    print(f"{asset:<15} {amt:>14,.0f} ${price:>8,.2f} ${value:>12,.0f} {value/total*100:>6.1f}%")

print(f"\nTotal value: ${total:,.0f}")
print(f"Diversification score: {treasury.diversification_score():.3f} (0=concentrated, 1=diversified)")
print(f"Monthly expenses: ${monthly_expenses:,.0f}")
print(f"Runway: {treasury.runway_months(monthly_expenses):.0f} months ({treasury.runway_months(monthly_expenses)/12:.1f} years)")

# Stress test
print(f"\nStress Test (native token -50%, ETH -30%):")
stressed = treasury.simulate_market_impact("GOV_TOKEN", {"GOV_TOKEN": -0.50, "ETH": -0.30})
print(f"  Treasury value: ${stressed:,.0f} ({(stressed/total - 1)*100:+.1f}%)")
print(f"  Runway: {stressed/monthly_expenses:.0f} months")

In [ ]:
# Treasury composition and stress test visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Composition pie chart
labels = list(treasury.holdings.keys())
values = [amt * price for amt, price in treasury.holdings.values()]
colors = ['#ff7f0e', '#1f77b4', '#2ca02c', '#17becf']
axes[0].pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title(f'Treasury Composition (${total/1e6:.0f}M total)')

# Right: Runway under different scenarios
scenarios = {
    'Current': {'GOV_TOKEN': 0, 'ETH': 0},
    'Bull (+50%)': {'GOV_TOKEN': 0.50, 'ETH': 0.50},
    'Bear (-30%)': {'GOV_TOKEN': -0.30, 'ETH': -0.30},
    'Crash (-70%)': {'GOV_TOKEN': -0.70, 'ETH': -0.70},
    'Death Spiral (-90%)': {'GOV_TOKEN': -0.90, 'ETH': -0.50},
}

scenario_names = list(scenarios.keys())
runways = [treasury.simulate_market_impact("GOV_TOKEN", s) / monthly_expenses
           for s in scenarios.values()]

bar_colors = ['green', 'darkgreen', 'orange', 'red', 'darkred']
bars = axes[1].barh(scenario_names, runways, color=bar_colors)
axes[1].axvline(x=12, color='black', linestyle='--', alpha=0.5, label='1 year')
axes[1].axvline(x=24, color='gray', linestyle='--', alpha=0.5, label='2 years')
axes[1].set_xlabel('Runway (months)')
axes[1].set_title('Treasury Runway Under Market Scenarios')
axes[1].legend()

for bar, runway in zip(bars, runways):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{runway:.0f}mo', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/treasury_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Treasury analysis complete.")
print("Key risk: Treasuries concentrated in native token lose value in bear markets.")

---
## 6. Governance Health Metrics

Quantitative metrics help assess whether governance is healthy or captured.

> **Definition: Gini Coefficient** -- A measure of inequality ranging from 0 (perfect equality) to 1 (maximum inequality). Applied to governance: measures how evenly voting power is distributed.

> **Definition: Nakamoto Coefficient** -- The minimum number of entities required to collectively control >50% of a system's decision-making power. Higher = more decentralized.

**Source:** Srinivasan, B. & Lee, L. (2017). "Quantifying Decentralization." news.earn.com

In [ ]:
def gini_coefficient(values: np.ndarray) -> float:
    """Calculate the Gini coefficient of a distribution."""
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals) / (n * np.sum(sorted_vals))) - (n + 1) / n


def nakamoto_coefficient(values: np.ndarray) -> int:
    """Calculate the Nakamoto coefficient (min entities for >50% control)."""
    sorted_desc = np.sort(values)[::-1]
    cumulative = np.cumsum(sorted_desc) / np.sum(sorted_desc)
    return int(np.searchsorted(cumulative, 0.5)) + 1


def voter_participation_rate(n_voters: int, n_holders: int) -> float:
    """Calculate voter participation rate."""
    return n_voters / n_holders if n_holders > 0 else 0


# Analyze governance metrics for several DAOs (synthetic)
np.random.seed(42)

dao_data = {
    "Uniswap": np.random.pareto(1.2, 5000) * 100000,
    "Compound": np.random.pareto(1.5, 2000) * 50000,
    "Aave": np.random.pareto(1.3, 3000) * 80000,
    "MakerDAO": np.random.pareto(1.8, 1500) * 30000,
    "ENS": np.random.pareto(1.0, 8000) * 200000,
}

participation_rates = {
    "Uniswap": 0.03, "Compound": 0.08, "Aave": 0.12,
    "MakerDAO": 0.15, "ENS": 0.05
}

print("=" * 70)
print("GOVERNANCE HEALTH METRICS")
print("=" * 70)

print(f"\n{'DAO':<12} {'Holders':>8} {'Gini':>6} {'Nakamoto':>10} {'Participation':>14} {'Health':>8}")
print("-" * 62)

for dao, dist in dao_data.items():
    gini = gini_coefficient(dist)
    naka = nakamoto_coefficient(dist)
    part = participation_rates[dao]
    
    # Simple health score
    health = (1 - gini) * 0.3 + min(naka / 20, 1) * 0.3 + min(part / 0.20, 1) * 0.4
    health_label = "Good" if health > 0.5 else "Fair" if health > 0.3 else "Poor"
    
    print(f"{dao:<12} {len(dist):>8,} {gini:>5.3f} {naka:>10} {part:>13.1%} {health_label:>8}")

In [ ]:
# Visualize governance metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dao_names = list(dao_data.keys())
ginis = [gini_coefficient(dao_data[d]) for d in dao_names]
nakas = [nakamoto_coefficient(dao_data[d]) for d in dao_names]
parts = [participation_rates[d] * 100 for d in dao_names]

# Top-left: Gini coefficients
colors = plt.cm.RdYlGn_r(np.array(ginis))
axes[0, 0].barh(dao_names, ginis, color=colors)
axes[0, 0].set_xlabel('Gini Coefficient')
axes[0, 0].set_title('Voting Power Inequality (lower = better)')
axes[0, 0].set_xlim(0, 1)

# Top-right: Nakamoto coefficients
naka_colors = plt.cm.RdYlGn(np.array(nakas) / max(nakas))
axes[0, 1].barh(dao_names, nakas, color=naka_colors)
axes[0, 1].set_xlabel('Nakamoto Coefficient')
axes[0, 1].set_title('Min Entities for 51% Control (higher = better)')

# Bottom-left: Participation rates
part_colors = plt.cm.RdYlGn(np.array(parts) / max(parts))
axes[1, 0].barh(dao_names, parts, color=part_colors)
axes[1, 0].set_xlabel('Participation Rate (%)')
axes[1, 0].set_title('Voter Participation')

# Bottom-right: Lorenz curves for all DAOs
for dao, dist in dao_data.items():
    sorted_vals = np.sort(dist)
    cumulative = np.cumsum(sorted_vals) / np.sum(sorted_vals)
    x = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    axes[1, 1].plot(x * 100, cumulative * 100, linewidth=2, label=dao)

axes[1, 1].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect equality')
axes[1, 1].set_xlabel('% of Token Holders')
axes[1, 1].set_ylabel('% of Voting Power')
axes[1, 1].set_title('Lorenz Curves')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/governance_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print("Governance health metrics visualized.")
print("Most DAOs show high Gini (inequality) and low participation.")

---
## Exercises

### Exercise 1: Rage Quit Mechanism

Implement a Moloch DAO-style "rage quit" mechanism where members can exit the DAO and withdraw their proportional share of the treasury if they disagree with a proposal.

**Hints:**
- Members can rage quit during the grace period (after vote, before execution)
- Rage quitting burns their shares and returns proportional treasury assets
- This protects minorities from hostile majority decisions

In [ ]:
class MolochDAO:
    """Moloch DAO with rage quit mechanism."""
    
    def __init__(self, members: Dict[str, float], treasury_value: float) -> None:
        """Initialize DAO with member shares and treasury."""
        self.shares = dict(members)
        self.treasury = treasury_value
        # YOUR CODE HERE
    
    def rage_quit(self, member: str) -> float:
        """Member exits and withdraws proportional treasury share."""
        # YOUR CODE HERE
        pass
    
    def simulate_contentious_vote(self, proposal_support: float) -> Dict:
        """Simulate a vote where losing side may rage quit."""
        # YOUR CODE HERE
        pass

### Exercise 2: Optimistic Governance

Implement optimistic governance where proposals pass automatically unless vetoed within a challenge period. This is used by protocols like Optimism's Token House.

**Hints:**
- Proposals auto-pass after a delay unless enough tokens vote to veto
- Veto threshold is typically higher than normal quorum
- Compare throughput vs regular governance

In [ ]:
class OptimisticGovernor:
    """Optimistic governance with veto mechanism."""
    
    def __init__(self, token_balances: Dict[str, float],
                 veto_threshold_pct: float = 10.0,
                 challenge_period: int = 7) -> None:
        """Initialize optimistic governor."""
        self.balances = dict(token_balances)
        self.total_supply = sum(token_balances.values())
        self.veto_threshold = self.total_supply * veto_threshold_pct / 100
        self.challenge_period = challenge_period
        # YOUR CODE HERE
    
    def propose(self, proposer: str, description: str) -> int:
        """Submit a proposal (auto-passes unless vetoed)."""
        # YOUR CODE HERE
        pass
    
    def veto(self, voter: str, proposal_id: int) -> None:
        """Vote to veto a proposal."""
        # YOUR CODE HERE
        pass
    
    def resolve(self, proposal_id: int) -> str:
        """Resolve proposal after challenge period."""
        # YOUR CODE HERE
        pass

### Exercise 3: Governance Participation Incentives

Model a system that rewards voters for participation (similar to Curve's vote-escrowed model) and analyze how incentives affect participation rates.

**Hints:**
- Voters earn rewards proportional to their participation
- Longer lock periods earn higher multipliers
- Model the tradeoff between participation incentives and governance quality

In [ ]:
class VoteIncentiveSystem:
    """Incentivized governance participation system."""
    
    def __init__(self, balances: Dict[str, float],
                 reward_pool_per_epoch: float = 100_000) -> None:
        """Initialize incentive system."""
        self.balances = dict(balances)
        self.reward_pool = reward_pool_per_epoch
        # YOUR CODE HERE
    
    def lock_tokens(self, user: str, amount: float, lock_periods: int) -> None:
        """Lock tokens for governance with time multiplier."""
        # YOUR CODE HERE
        pass
    
    def distribute_rewards(self, voters: List[str]) -> Dict[str, float]:
        """Distribute rewards to voters proportional to locked power."""
        # YOUR CODE HERE
        pass
    
    def simulate_participation_impact(self, epochs: int = 52) -> Dict:
        """Simulate how incentives affect participation over time."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Full governance proposal lifecycle: propose, vote, queue, execute
- [x] Token-weighted, quadratic, and conviction voting mechanisms and their tradeoffs
- [x] Flash loan attacks, vote buying, and voter apathy as governance attack vectors
- [x] Vote delegation systems and power concentration analysis
- [x] DAO treasury management, diversification, and runway analysis
- [x] Governance health metrics: Gini coefficient, Nakamoto coefficient, participation rates

### Key Takeaways
1. **Token-weighted voting is plutocratic** -- quadratic voting and conviction voting offer alternatives
2. **Snapshot-based voting prevents flash loan attacks** -- always use historical balance snapshots
3. **Low participation is the norm** -- most DAOs see <10% turnout, making them vulnerable
4. **Delegation concentrates power** -- a few delegates often control majority voting power
5. **Treasury concentration is risky** -- DAOs holding mostly native tokens face correlated value declines
6. **Governance is a spectrum** -- from full on-chain democracy to optimistic/veto models

### Further Reading
- Buterin, V. (2021). "Moving beyond coin voting governance." vitalik.eth.limo
- Aragon. (2023). "Governance Framework Documentation." aragon.org
- DeepDAO. (2024). "DAO Governance Analytics." deepdao.io

### Next Steps
- [Notebook 13: Stablecoin Analysis](13-stablecoin-analysis.ipynb) -- Stablecoin mechanisms and risk
- [Section 08: DAO Governance](../sections/08-dao-governance.md) -- Governance theory and case studies